In [ ]:
EXP_NAME = "19MAY_cevaehe_9"
TRAINING_DATASET = "heart_disease_train.csv"
TEST_DATASET = "heart_disease_test.csv"
FEATURE_MAP = "scm/uci_scm_config_simplified_2.json"
SEED = 4

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from notebooks.analysis_utils import mutual_info_grouped, plot_cat_feature, plot_cont_feature, plot_cf_cont_feature_comparison, plot_cf_cat_feature_comparison

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

In [ ]:
test_latents = pd.read_csv(f'{PROJECT_ROOT}/data/{EXP_NAME}/test_latent_space.csv')
test_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/data/{EXP_NAME}/test_counterfactuals.csv')
training_latents = pd.read_csv(f'{PROJECT_ROOT}/data/{EXP_NAME}/train_latent_space.csv')
training_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/data/{EXP_NAME}/train_counterfactuals.csv')
test_dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{TEST_DATASET}')
training_dataset = pd.read_csv(f'{PROJECT_ROOT}/data/{TRAINING_DATASET}')

with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
training_df = training_latents.merge(training_dataset, how="inner", left_on="patient_index", right_index=True)
training_df = training_df.merge(training_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
training_df.drop('sample_index_cf', axis=1, inplace=True)

test_df = test_latents.merge(test_dataset, how="inner", left_on="patient_index", right_index=True)
test_df = test_df.merge(test_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
test_df.drop('sample_index_cf', axis=1, inplace=True)

In [ ]:
desc_cols = [f['name'] for f in feature_map['desc']]
desc_cf_cols = [f"{f['name']}_cf" for f in feature_map['desc']]
latent_cols = [col for col in training_latents.columns if 'u_desc_' in col]
sens_col = feature_map['sens'][0]['name']
target_col = feature_map['target']['name']

# Latent space

## Mutual Information with $X_{sens}$

### Training dataset

In [ ]:
mutual_info_grouped(
  training_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=400,
  seed=SEED
)

### Test dataset

In [ ]:
mutual_info_grouped(
  test_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=100,
  seed=SEED
)

# Counterfactuals

In [ ]:
training_df.head()

In [ ]:
for feature in feature_map['desc']:
  col_name = feature['name']
  cf_col_name = feature['name'] + "_cf"
  group_0_mask = training_df[sens_col] == 0
  group_1_mask = training_df[sens_col] == 1
  if feature['type'] == "continuous":
    plot_cf_cont_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)
  else:
    plot_cf_cat_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)